# 01 -- Dataset setup and registry check

Verifies `data/registry.py`'s `DATASET_REGISTRY` entries against the actual raw CSV headers on disk for every `active_datasets` entry, before running anything expensive. Run this after any change to `config/default.yaml`'s `data.active_datasets` or to `registry.py` itself.


## Setup

Run this cell first. It's the ONLY cell you should need to edit: change
`CONFIG_OVERRIDES` (a list of `--set key.path=value` style dotted overrides,
same syntax as `training.run`'s CLI) to narrow `data.active_datasets`,
switch `architecture`, point at a different Drive folder, etc.


In [ ]:
# ---- Single config cell: this is the only cell you should need to edit ----
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, os
    REPO_DIR = '/content/dataset_moe_nids'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/selimsidan/dataset_moe_nids', REPO_DIR], check=True)
    %cd $REPO_DIR
    %pip install -q -r requirements.txt

CONFIG_PATH = 'config/default.yaml'

# Dotted --set overrides, same syntax as training.run's CLI. Narrow
# data.active_datasets to 2-3 datasets here for a fast iteration cycle --
# every downstream module (registry, harmonizer, expert-bank sizing,
# checkpoints) adapts automatically, no other code changes needed.
CONFIG_OVERRIDES = [
    # 'data.active_datasets=[NF-UNSW-NB15-v3,NF-BoT-IoT-v3]',
    # 'architecture=moe_dataset_soft',
    # 'training.device=cuda',
]

from training.config import load_config
config = load_config(CONFIG_PATH, CONFIG_OVERRIDES)
print('run_name:', config['run_name'])
print('architecture:', config['architecture'])
print('active_datasets:', config['data']['active_datasets'])
print('checkpoint_dir:', config['training']['checkpoint_dir'])


## Inspect each active dataset's registered schema

In [ ]:
from data.registry import get_spec

for name in config['data']['active_datasets']:
    spec = get_spec(name)
    print(f"\n=== {name} ===")
    print('kind:', spec.kind, '| label_col:', spec.label_col, '| benign_label:', spec.benign_label)
    print('paths:', spec.paths)
    print(f'feature_alias ({len(spec.feature_alias)} features):', spec.feature_alias)


## Cross-check registered aliases against real CSV headers on disk

Flags any `feature_alias` raw column name that ISN'T found in the file's actual header row -- the fastest way to catch a stale/incorrect mapping before a real run.

In [ ]:
import os
import pandas as pd

for name in config['data']['active_datasets']:
    spec = get_spec(name)
    candidates = [p for p in spec.paths if os.path.isfile(p)] or (
        [os.path.join(d, f) for d in spec.paths if os.path.isdir(d) for f in os.listdir(d) if f.endswith('.csv')][:1]
    )
    if not candidates:
        print(f'[{name}] NO FILE FOUND on disk -- check data/paths.py NIDS_DRIVE_BASE / registry paths.')
        continue
    header = pd.read_csv(candidates[0], nrows=0).columns.tolist()
    header = [c.strip() for c in header]
    missing = [raw for raw in spec.feature_alias.values() if raw not in header]
    status = 'OK' if not missing else f'MISSING {missing}'
    print(f'[{name}] {candidates[0]}: {status}')
